# 3.2 - History of GenAI

**Module 3 - GenAI Foundations** | Netsetos GenAI Engineering

This notebook works through History of GenAI hands-on: The GAN concept in 30 lines; Self-attention in 15 lines; Generation across eras; The diffusion concept.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

One install and a seeded NumPy import. Every milestone in this notebook is pure NumPy math (with one optional Gemini call later), and several of them draw random numbers - so we fix the seed here to make runs reproducible.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install -q google-genai
# Uncomment the next line if needed:
# !pip install numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

This is environment prep, not a milestone. It installs the Gemini SDK for the later generation cell and seeds NumPy's global random generator so any run you compare against the walkthrough lines up.

**How the code works - step by step**
- **`!pip install -q google-genai`** - installs the current unified Google Gemini SDK, used only by the generation-across-eras cell later.
- **`import numpy as np`** - NumPy is the only math dependency; every milestone cell is plain array math.
- **`np.random.seed(42)`** - seeds the global RNG so seeded draws are deterministic.

**In one line:** install the Gemini SDK, import NumPy, fix the seed.

**What you'll see:** (no output - environment prep)

## 1 - The GAN concept in 30 lines

A GAN pits two networks against each other: a generator that fakes data and a discriminator that tries to catch the fakes. This cell strips that idea down to numbers - the generator tries to match a target distribution (mean 5, std 1) starting from a bad guess - so you can watch the competition converge without any neural network.

In [ ]:
# =============================================
# GAN CONCEPT - The Forger vs The Detective
# This shows the IDEA, not a full GAN.
# A real GAN uses neural networks - this uses
# simple math to demonstrate the competition.
# =============================================

import numpy as np

# The "real" data: numbers drawn from a normal distribution
# (mean=5, std=1). The generator must learn to produce
# numbers that look like they came from this distribution.
real_mean, real_std = 5.0, 1.0

# Generator starts with a bad guess
gen_mean, gen_std = 0.0, 3.0
learning_rate = 0.1
np.random.seed(0)  # reproducible run

print("GAN Training - Generator learns to match real data:\n")

for epoch in range(20):
    # Generator produces fake samples
    fake_samples = np.random.normal(gen_mean, abs(gen_std), 100)
    real_samples = np.random.normal(real_mean, real_std, 100)
    
    # Discriminator's "score": how different are fake vs real?
    mean_diff = np.mean(real_samples) - np.mean(fake_samples)
    std_diff = np.std(real_samples) - np.std(fake_samples)
    
    # Generator adjusts to fool the discriminator
    gen_mean += learning_rate * mean_diff
    gen_std += learning_rate * std_diff
    
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:2d}: gen_mean={gen_mean:.2f} (target={real_mean}) | gen_std={abs(gen_std):.2f} (target={real_std})")

print(f"\n  Final: gen_mean={gen_mean:.2f}, gen_std={abs(gen_std):.2f}")
print(f"  Target: mean={real_mean}, std={real_std}")
print(f"\n  The generator learned to produce numbers that look real!")
print(f"  In a real GAN, replace numbers with images → photorealistic faces.")
# Output:  (seeded - deterministic; converges toward the target)
# GAN Training - Generator learns to match real data:
#   Epoch  0: gen_mean=0.49 (target=5.0) | gen_std=2.80 (target=1.0)
#   Epoch  5: gen_mean=2.38 (target=5.0) | gen_std=2.08 (target=1.0)
#   Epoch 10: gen_mean=3.44 (target=5.0) | gen_std=1.63 (target=1.0)
#   Epoch 15: gen_mean=4.06 (target=5.0) | gen_std=1.37 (target=1.0)
#   Final: gen_mean=4.38, gen_std=1.23  (target: mean=5.0, std=1.0)

**Explanation**

This is a demonstration of the adversarial idea, not a real GAN - there are no neural networks, just summary statistics standing in for the two players. The generator holds a guessed mean and std, and each epoch it nudges them toward the real data by the gap the 'discriminator' reports; read it as a feedback loop that closes the distance between fake and real distributions.

**How the code works - step by step**
- **`real_mean, real_std`** - defines the real distribution the generator must learn to imitate (mean 5, std 1).
- **`gen_mean, gen_std`** - the generator's deliberately bad starting guess (mean 0, std 3).
- **`fake_samples` / `real_samples`** - each epoch draws 100 numbers from the current generator and 100 from the real distribution.
- **`mean_diff` / `std_diff`** - the discriminator's 'score': how far the fake batch's mean and spread sit from the real batch's.
- **`gen_mean += ... / gen_std += ...`** - the generator moves its parameters toward the real ones by `learning_rate` times that gap.
- **print every 5th epoch + final line** - shows the guess closing on the target over 20 epochs.

**In one line:** measure how far fake is from real, step toward real, repeat.

**What you'll see:** A seeded, deterministic training log: gen_mean climbs from 0.49 toward the target 5.0 (ending around 4.38) and gen_std falls from 2.80 toward 1.0 (ending around 1.23), followed by a note that in a real GAN the numbers would be image pixels.

## 2 - Self-attention in 15 lines

Self-attention is the one mechanism that made Transformers work: every word looks at every other word and scores how relevant each is to it. This cell builds the full query-key-softmax pipeline over a six-word sentence with random embeddings, so you can see the exact matrix math that scales up to GPT.

In [ ]:
# =============================================
# SELF-ATTENTION - The Core of Transformers
# Every word looks at every other word and asks:
# "How relevant are you to me?"
# =============================================

import numpy as np

# Sentence: "The cat sat on the mat"
# Each word is represented as a vector (simplified to 4 dims)
words = ["The", "cat", "sat", "on", "the", "mat"]
np.random.seed(42)
embeddings = np.random.randn(6, 4)  # 6 words × 4 dimensions

# Self-attention: how much should each word "attend to" every other word?
# Step 1: Compute attention scores (dot product of all pairs)
scores = embeddings @ embeddings.T  # 6×6 matrix

# Step 2: Scale (prevents huge numbers)
d_k = embeddings.shape[1]
scores = scores / np.sqrt(d_k)

# Step 3: Softmax (convert to probabilities)
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attention = softmax(scores)

# Show which words "cat" pays attention to
print("Self-Attention: which words does 'cat' attend to?\n")
cat_idx = 1
for i, (word, weight) in enumerate(zip(words, attention[cat_idx])):
    bar = "█" * int(weight * 40)
    print(f"  {word:4s} → {weight:.3f} {bar}")

print(f"\n  With RANDOM embeddings these weights are arbitrary.")
print(f"  In a trained Transformer, 'cat' would attend strongly to 'sat'")
print(f"  (what did the cat do?) and 'mat' (where is the cat?).")
print(f"\n  This is the ENTIRE secret of Transformers.")
print(f"  Scale this to 96 layers and 175 billion parameters → GPT-3.")
# Output:  (seeded - deterministic)
# Self-Attention: which words does 'cat' attend to?
#   The  -> 0.303 ############
#   cat  -> 0.521 ####################
#   sat  -> 0.061 ##
#   on   -> 0.027 #
#   the  -> 0.033 #
#   mat  -> 0.056 ##
#   (random embeddings -> arbitrary weights; training is what makes 'cat'->'sat' strong)

**Explanation**

This is a from-scratch trace of the attention computation, not a trained model - the embeddings are random, so the resulting weights are arbitrary and the cell says so. The point is the shape of the calculation: dot-product every pair of word vectors, scale, softmax into probabilities that sum to 1, and read off one word's attention row.

**How the code works - step by step**
- **`embeddings = np.random.randn(6, 4)`** - represents each of the 6 words as a 4-dimensional vector (random, untrained).
- **`scores = embeddings @ embeddings.T`** - the 6x6 matrix of dot products: every word's raw relevance to every other word.
- **`scores / np.sqrt(d_k)`** - scales by the square root of the dimension to keep the numbers from blowing up before softmax.
- **`softmax(...)`** - the numerically-stable softmax (subtract the row max, exponentiate, normalize) turns each row into probabilities that sum to 1.
- **loop over `attention[cat_idx]`** - prints the attention weights for the word 'cat' as text bars.

**In one line:** dot-product all word pairs -> scale -> softmax -> read one word's attention row.

**What you'll see:** A seeded, deterministic bar chart of what 'cat' attends to (e.g. cat 0.521, The 0.303, sat 0.061), followed by a note that with random embeddings these weights are arbitrary and that training is what would make 'cat' attend strongly to 'sat'.

## 3 - Generation across eras

Same prompt - 'describe a sunset over Hyderabad' - run through four eras of generative models. The first three eras are hard-coded illustrations of what each generation could produce; the fourth makes a live Gemini call so you can see the gap for real.

> **Needs a Gemini API key** (set GEMINI_API_KEY) for the live line; without it the cell prints a canned fallback.

In [ ]:
# =============================================
# GENERATION ACROSS ERAS
# Simulate what each era could produce for the
# same task: "Describe a sunset over Hyderabad."
# =============================================

import os
import numpy as np
from google import genai

# Current unified SDK (pip install google-genai)
api_key = os.environ.get("GEMINI_API_KEY")

print("Task: 'Describe a sunset over Hyderabad'\n")

# Era 1: Boltzmann Machine (1985) - random noise
print("1985 - Boltzmann Machine:")
random_words = np.random.choice(["sun","red","sky","hot","big","the","cloud","bird"], 8)
print(f"  {' '.join(random_words)}")
print("  (random word soup - no grammar, no meaning)\n")

# Era 2: Simple RNN (~2015) - grammatical but shallow
print("2015 - Simple RNN:")
print("  The sun is red. The sky is big. The sun is hot.")
print("  (grammatical but repetitive, no understanding)\n")

# Era 3: LSTM (~2017) - coherent but generic
print("2017 - LSTM:")
print("  The sunset was beautiful. The sky turned orange")
print("  and red. It was a nice evening in the city.")
print("  (coherent but generic - could be any city)\n")

# Era 4: Modern Transformer (today) - rich, specific, creative
print("Today - Gemini (Transformer):")
if api_key:
    client = genai.Client(api_key=api_key)
    resp = client.models.generate_content(
        model="gemini-3-flash",
        contents=(
            "Describe a sunset over Hyderabad in 3 vivid sentences. "
            "Mention specific landmarks like Charminar or Hussain Sagar."
        ),
    )
    print(f"  {resp.text.strip()}")
else:
    print("  [set GEMINI_API_KEY to see the live Gemini output]")
    print("  Charminar silhouetted against a tangerine sky as Hussain Sagar")
    print("  mirrors the dying light over the old city.")
print("  (specific, vivid, culturally aware, creative)\n")

print("═" * 50)
print("40 years of progress in one comparison.")
print("Same task. Dramatically different capability.")
# Output:  (the live Gemini line varies by run)
# Task: 'Describe a sunset over Hyderabad'
# 1985 - Boltzmann:   sun red sky the big cloud bird hot   (random word soup)
# 2015 - Simple RNN:  The sun is red. The sky is big.       (grammatical, shallow)
# 2017 - LSTM:        The sunset was beautiful...           (coherent, generic)
# Today - Gemini:     Charminar silhouetted against a tangerine sky as Hussain Sagar
#                     mirrors the dying light over the old city.  (vivid, specific)

**Explanation**

This is a side-by-side capability comparison, mostly hard-coded strings, with one real API call at the end. It builds each era's output in reading order - random word soup (1985), grammatical-but-shallow (2015 RNN), coherent-but-generic (2017 LSTM), then a genuine Gemini response - to make '40 years of progress' concrete on a single task.

**How the code works - step by step**
- **`api_key = os.environ.get('GEMINI_API_KEY')`** - reads the key; its presence decides whether the last era runs live.
- **Era 1 (Boltzmann)** - `np.random.choice` samples 8 words at random: deliberate word soup.
- **Era 2 (RNN)** - a hard-coded grammatical-but-repetitive line.
- **Era 3 (LSTM)** - a hard-coded coherent-but-generic line.
- **Era 4 (Gemini)** - if the key exists, builds a `genai.Client` and calls `generate_content` on `gemini-3-flash` with a landmark-specific prompt; otherwise prints a canned Charminar/Hussain Sagar fallback.
- **closing prints** - frame the four outputs as 40 years of progress on one task.

**In one line:** run one prompt through four eras - three canned, one live - and compare.

**What you'll see:** The four era outputs stacked top to bottom; the 1985 line varies (random word choice) and the live Gemini line varies by run, while the RNN and LSTM lines are fixed. With no key set, era 4 shows the canned Charminar fallback.

## 4 - The diffusion concept

Diffusion generates images with one deceptively simple loop: take a signal, destroy it with noise, then learn to remove the noise step by step. This cell runs that loop in 1D on a sine wave - the 'image' - so you can watch the forward corruption and the reverse recovery as text bars.

In [ ]:
# =============================================
# DIFFUSION - Add Noise, Remove Noise, Create
# Simplified 1D version of the concept behind
# Stable Diffusion, DALL-E, and Midjourney.
# =============================================

import numpy as np

# "Image" = a simple signal (sine wave)
np.random.seed(42)
original = np.sin(np.linspace(0, 2 * np.pi, 20))

print("Diffusion Process - Adding then Removing Noise:\n")

# Forward process: gradually add noise (destroy the signal)
print("Forward (add noise):")
noisy = original.copy()
noise_levels = [0.0, 0.2, 0.5, 1.0, 2.0]
for noise in noise_levels:
    noisy = original + np.random.randn(20) * noise
    signal_quality = max(0, int((1 - noise/2) * 10))
    bar = "█" * signal_quality + "░" * (10 - signal_quality)
    print(f"  noise={noise:.1f}: [{bar}] {'clear signal' if noise < 0.3 else 'noisy' if noise < 1 else 'pure noise'}")

print(f"\nReverse (denoise) - this is what Stable Diffusion learns:")
# Reverse: the model learns to predict and remove noise at each step
reconstructed = noisy.copy()
for step in range(5):
    # In reality: a neural network predicts the noise to remove
    # Here: we cheat by moving toward the original signal
    reconstructed = reconstructed * 0.6 + original * 0.4
    similarity = 1 - np.mean(np.abs(reconstructed - original))
    bar = "█" * int(similarity * 10) + "░" * (10 - int(similarity * 10))
    print(f"  step {step+1}: [{bar}] similarity={similarity:.2f}")

print(f"\n  Start: pure noise → 5 denoising steps → recovered signal!")
print(f"  In Stable Diffusion: noise is random pixels, signal is an image.")
print(f"  Text prompt tells the denoiser WHAT image to create.")
# Output:  (seeded - deterministic)
# Forward (add noise):
#   noise=0.0: [##########] clear signal
#   noise=0.5: [#######...] noisy
#   noise=2.0: [..........] pure noise
# Reverse (denoise) - what Stable Diffusion learns:
#   step 1: [###.......] similarity=0.34
#   step 3: [#######...] similarity=0.76
#   step 5: [#########.] similarity=0.92

**Explanation**

This is a simplified 1D stand-in for the diffusion process, not a trained denoiser - the reverse step openly 'cheats' by blending back toward the known original instead of predicting noise with a network. It has two halves: a forward pass that adds increasing noise until the sine wave is static, and a reverse pass that recovers it, mirroring what Stable Diffusion learns.

**How the code works - step by step**
- **`original = np.sin(...)`** - a clean 20-point sine wave stands in for an image.
- **forward loop over `noise_levels`** - adds Gaussian noise of growing size and prints a signal-quality bar that decays from clear to pure static.
- **reverse loop `range(5)`** - each step does `reconstructed * 0.6 + original * 0.4`, a stand-in for a network predicting and subtracting noise, and prints a rising similarity score.
- **closing prints** - map the 1D toy onto real Stable Diffusion: noise is random pixels, the recovered signal is an image, and the text prompt tells the denoiser what to make.

**In one line:** add noise until the signal is static, then blend back toward it step by step.

**What you'll see:** A seeded, deterministic two-part readout: a forward section where the quality bar decays from clear signal to pure noise, then a reverse section where similarity climbs step by step (roughly 0.34 -> 0.76 -> 0.92) as the sine wave is recovered.

Four tiny scripts, four eras: a competition that converges (GAN), a table of pairwise relevance scores (self-attention), a side-by-side of what each era could say (generation), and a destroy-then-recover loop (diffusion). None of them is a real trained model - each strips a landmark idea down to a dozen lines of NumPy so you can watch the mechanism run. You will build the real versions later: the Transformer from scratch in Module 4, diffusion image generation in Module 6, and the API-driven multimodal frontier throughout the rest of the course.